# PROMPT ENGINEERING AND RAG
## SESSION 2

Assignment questions and answers


## Question 1

Task 1: Structured Prompt Builder for Food Delivery
Chatbot

Build a Python script that constructs and validates a role-based,
structured prompt for a food-delivery customer-support chatbot and
prints the formatted prompt to the console.

Requirements: - Define a system prompt assigning the LLM the role of a
food-delivery support agent, setting a professional and helpful tone,
and specifying that responses must not exceed 80 words. - Define a
formatted user-message template accepting customer_name, order_id, and
issue_type. Populate it with at least two different test cases and print
the complete system and user prompt pair for each. - Add a validation
function that raises ValueError with a clear message if issue_type is
not one of: ‘late delivery’, ‘missing item’, or ‘wrong item’. - Wrap
prompt construction in try-except, catch ValueError, and print a
user-friendly error message without crashing.

**Answer:**

```python
# System prompt
SYSTEM_PROMPT = (
    "You are a professional and helpful food-delivery customer support agent. "
    "Assist the customer with their issue. Keep your response under 80 words."
)

VALID_ISSUES = {"late delivery", "missing item", "wrong item"}


def validate_issue(issue_type: str) -> None:
    if issue_type.lower() not in VALID_ISSUES:
        raise ValueError(
            f"Invalid issue type: '{issue_type}'. Allowed types: {', '.join(VALID_ISSUES)}."
        )


def build_prompt(customer_name: str, order_id: str, issue_type: str) -> None:
    try:
        validate_issue(issue_type)
        user_message = (
            f"Customer Name: {customer_name}\n"
            f"Order ID: {order_id}\n"
            f"Issue: {issue_type}"
        )

        print("=" * 40)
        print(f"SYSTEM PROMPT:\n{SYSTEM_PROMPT}\n")
        print(f"USER PROMPT:\n{user_message}")
        print("=" * 40 + "\n")

    except ValueError as e:
        print(f"[Error] Could not generate prompt: {e}\n")


# Test Cases
test_cases = [
    {"customer_name": "Alice Smith", "order_id": "ORD-101", "issue_type": "late delivery"},
    {"customer_name": "Bob Jones", "order_id": "ORD-102", "issue_type": "missing item"},
    {"customer_name": "Charlie Brown", "order_id": "ORD-103", "issue_type": "damaged package"},  # Triggers error
]

for case in test_cases:
    build_prompt(**case)
```

## Question 2

Task 2: Few-Shot Complaint Classifier Prompt Builder

Build a Python function that constructs a few-shot prompt for
classifying food-delivery complaints into predefined categories and
returns the formatted prompt string ready for an LLM API.

Requirements: - Define a list of at least four labelled Input / Output
examples covering ‘Late Delivery’, ‘Wrong Item’, ‘Missing Item’, and
‘Poor Quality’. - Write build_few_shot_prompt(complaint_text) to insert
the labelled examples before the new complaint and a blank Output line
for the LLM to complete. - Write add_example(text, label) to append a
new labelled pair to the examples list. Call it once with a new example,
then call build_few_shot_prompt again and confirm the new example
appears in the printed output. - Test the prompt builder with at least
two different complaint strings and print each resulting prompt, showing
consistent structure.

**Answer:**

```python
# Initial labeled examples
examples = [
    {"input": "The food took two hours to arrive and was cold.", "output": "Late Delivery"},
    {"input": "I ordered a pepperoni pizza but received a veggie burger.", "output": "Wrong Item"},
    {"input": "The combo meal was missing the french fries and soda.", "output": "Missing Item"},
    {"input": "The soup was completely spilled inside the bag and inedible.", "output": "Poor Quality"}
]

def build_few_shot_prompt(complaint_text):
    prompt_lines = ["Classify the customer complaint into one of the categories: Late Delivery, Wrong Item, Missing Item, Poor Quality.\n"]
    for ex in examples:
        prompt_lines.append(f"Input: {ex['input']}\nOutput: {ex['output']}\n")
    prompt_lines.append(f"Input: {complaint_text}\nOutput:")
    return "\n".join(prompt_lines)

def add_example(text, label):
    examples.append({"input": text, "output": label})

# Test 1: Generate prompt with initial examples
print("--- TEST 1 ---")
prompt_1 = build_few_shot_prompt("The driver was 45 minutes past the estimated time.")
print(prompt_1)

# Add a new example
add_example("The chicken was completely raw in the middle.", "Poor Quality")

# Test 2: Generate prompt showing newly added example
print("\n--- TEST 2 (After add_example) ---")
prompt_2 = build_few_shot_prompt("I only received 1 drink instead of the 3 I paid for.")
print(prompt_2)
```

## Question 3

Task 3: Semantic Search Over Restaurant FAQs

Build a Python script that encodes restaurant FAQ entries into
embeddings using sentence-transformers and retrieves the most
semantically relevant FAQ for a user query using a FAISS index.

Requirements: - Define at least six FAQ strings covering topics such as
delivery time, order cancellation, refund eligibility, item
substitution, and contacting support. Generate embeddings using
SentenceTransformer(‘all-MiniLM-L6-v2’). - Build a FAISS IndexFlatL2
index from FAQ embeddings converted to a float32 NumPy array, and
confirm that the index size matches the number of FAQs. - Write
search_faq(query, k=2) to encode the query, search the index, and return
the top-k FAQ strings with their L2 distances. - Test with at least two
queries using different words from the stored FAQs (for example, ‘How do
I get money back?’ versus an FAQ phrased as ‘Refund policy’) and print
the results to demonstrate semantic matching beyond exact keyword
overlap.

**Answer:**

```python
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Define at least six FAQ entries
faqs = [
    "Standard delivery typically takes 30 to 45 minutes depending on traffic.",
    "Orders can be cancelled within 5 minutes of placing them through the app.",
    "Refund eligibility requires reporting missing or damaged items within 24 hours.",
    "If an ingredient is unavailable, we substitute it with a similar item of equal value.",
    "You can contact our customer support team via the in-app chat or phone hotline.",
    "We accept all major credit cards, debit cards, and digital wallet payments.",
]

# 2. Generate embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
faq_embeddings = model.encode(faqs)

# 3. Build FAISS IndexFlatL2
dimension = faq_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
faq_embeddings_np = np.array(faq_embeddings, dtype=np.float32)
index.add(faq_embeddings_np)

# Confirm index size
assert index.ntotal == len(
    faqs
), f"Index size {index.ntotal} does not match FAQ count {len(faqs)}"
print(f"Index successfully built with {index.ntotal} entries.\n")


# 4. Search function
def search_faq(query, k=2):
    query_vector = model.encode([query])
    query_vector_np = np.array(query_vector, dtype=np.float32)
    distances, indices = index.search(query_vector_np, k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append((faqs[idx], float(dist)))
    return results


# 5. Test queries
test_queries = [
    "How do I get my money back?",
    "How fast will my food arrive?",
]

for query in test_queries:
    print(f"Query: '{query}'")
    matches = search_faq(query, k=2)
    for rank, (faq, dist) in enumerate(matches, 1):
        print(f"  {rank}. [Distance: {dist:.4f}] {faq}")
    print()
```

## Question 4

Task 4: RAG-Powered Food Delivery Policy Q&A Pipeline

Build a Python script implementing a minimal RAG pipeline: chunk a
policy document, embed the chunks, retrieve relevant context, and
assemble a structured final prompt.

Requirements: - Define a multi-paragraph food-delivery policy text of at
least 300 words covering refund rules, delivery windows, and order
cancellation. Split it into overlapping chunks of approximately 100
words with a 20-word overlap and print the total chunk count. - Generate
sentence-transformer embeddings for all chunks, store them in a FAISS
IndexFlatL2 index, and implement retrieve(query, k=3) returning the
top-k relevant chunk strings. - Write build_rag_prompt(query,
retrieved_chunks) to construct a prompt in this order: system role
instruction, numbered retrieved context blocks, user question, and
explicit instruction to answer only from the provided context and say ‘I
don’t know’ if the answer is absent. - Demonstrate the end-to-end
pipeline with a hardcoded question such as ‘What is the refund policy
for missing items?’ Print retrieved chunks, assembled prompt, and a
placeholder simulated answer string.

**Answer:**

```python
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Policy document (>300 words)
policy_text = """
Food Delivery Policy and Customer Service Guidelines

1. Delivery Windows and Timeframes
We strive to deliver orders within the estimated timeframe shown at checkout, typically between 30 to 45 minutes depending on traffic and weather conditions. During peak hours or inclement weather, delivery windows may be extended up to 60 minutes. Customers will receive real-time updates via SMS and app notifications regarding courier location. If an order is delayed by more than 30 minutes beyond the maximum estimated delivery window, the customer qualifies for automatic delivery fee credits applied directly to their account for future orders.

2. Order Cancellation Policy
Customers may cancel their food orders free of charge within 5 minutes of order placement. Once a restaurant accepts the order and begins kitchen preparation, cancellations are subject to a partial fee of 50% of the subtotal to compensate the restaurant for ingredients used. Once a delivery courier has been dispatched and picks up the meal, orders can no longer be canceled, and no refunds will be issued for cancellations attempted at this stage. Scheduled orders can be modified or canceled up to two hours prior to the scheduled delivery time without penalty.

3. Refund Rules and Missing Items
If an item is missing from your order, report the issue through the app within 24 hours of delivery. We will issue a direct refund to the original payment method or credit account for the value of the missing item. For incorrect orders or severe food quality issues, customers must submit a photograph of the received items within 24 hours. Full refunds, partial refunds, or meal redeliveries will be authorized based on the investigation outcome. Delivery fees and tips are non-refundable unless the entire order is missing or completely damaged during transit. Refund processing typically takes 3 to 5 business days depending on the financial institution.
"""

# 2. Chunking with overlap (~100 words, ~20 words overlap)
words = policy_text.split()
chunk_size = 100
overlap = 20
step = chunk_size - overlap

chunks = []
for i in range(0, len(words), step):
    chunk = " ".join(words[i : i + chunk_size])
    chunks.append(chunk)
    if i + chunk_size >= len(words):
        break

print(f"Total Chunks: {len(chunks)}\n")

# 3. Embeddings and FAISS Index
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks, convert_to_numpy=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)


def retrieve(query, k=3):
    query_vector = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vector, k)
    return [chunks[idx] for idx in indices[0]]


# 4. Prompt Assembly
def build_rag_prompt(query, retrieved_chunks):
    context_str = "\n\n".join(
        [f"[{i+1}] {chunk}" for i, chunk in enumerate(retrieved_chunks)]
    )
    prompt = (
        "System: You are a customer support AI assistant for a food delivery service.\n\n"
        f"Context:\n{context_str}\n\n"
        f"User Question: {query}\n\n"
        "Instructions: Answer the question using ONLY the provided context blocks above. "
        "If the answer cannot be found in the context, respond with 'I don't know'."
    )
    return prompt


# 5. End-to-End Pipeline Demonstration
query = "What is the refund policy for missing items?"
retrieved = retrieve(query, k=3)
assembled_prompt = build_rag_prompt(query, retrieved)
simulated_answer = (
    "If an item is missing, report it via the app within 24 hours of delivery to receive "
    "a direct refund to your original payment method or account credit."
)

print("--- RETRIEVED CHUNKS ---")
for i, c in enumerate(retrieved, 1):
    print(f"\nChunk {i}:\n{c}")

print("\n--- ASSEMBLED PROMPT ---")
print(assembled_prompt)

print("\n--- SIMULATED ANSWER ---")
print(simulated_answer)
```